In [93]:
import requests
from bs4 import BeautifulSoup
import re
import json
import pandas as pd
import numpy as np

In [94]:
url_list_of_countries = 'https://simple.wikipedia.org/wiki/List_of_countries'
url_list_of_areas = 'https://en.wikipedia.org/wiki/List_of_countries_and_dependencies_by_area'
url_list_of_population = 'https://en.wikipedia.org/wiki/List_of_countries_and_dependencies_by_population'
url_list_of_languages = 'https://en.wikipedia.org/wiki/List_of_official_languages'
url_list_of_location_and_capital = 'https://en.wikipedia.org/wiki/List_of_national_capitals'
url_list_of_minimum_wages = 'https://en.wikipedia.org/wiki/List_of_countries_by_minimum_wage'
url_list_of_english_speaking_by_percentage = 'https://en.wikipedia.org/wiki/List_of_countries_by_English-speaking_population'
url_list_fertility_rate_2024 = 'https://en.wikipedia.org/wiki/List_of_countries_by_total_fertility_rate#Country_ranking_by_non-governmental_organizations'
url_list_country_iso_2 = 'https://en.wikipedia.org/wiki/List_of_ISO_3166_country_codes'
url_list_median_age = 'https://en.wikipedia.org/wiki/List_of_countries_by_median_age'
url_list_inflation_rate = 'https://en.wikipedia.org/wiki/List_of_countries_by_inflation_rate'
url_list_debt = 'https://en.wikipedia.org/wiki/List_of_countries_by_external_debt'
url_list_electricity_consumption = 'https://en.wikipedia.org/wiki/List_of_countries_by_electricity_consumption'
url_list_average_elevation = 'https://en.wikipedia.org/wiki/List_of_countries_by_average_elevation'
countries: dict = {}

In [95]:
def create_json_friendly_format_structure_with_all_countries(url: str, json_friendly_format: dict) -> dict:
    response = requests.get(url)

    if response.status_code != 200:
        raise ValueError('No response from the server!')
    
    soup = BeautifulSoup(response.text, 'html.parser')
    country_container_parent = soup.find('div', class_='mw-content-ltr mw-parser-output')
    country_container = country_container_parent.find_all('p')[1:]
    country_list =  [x.text.strip() for x in country_container]

    for sub_countries in country_list:
        countries_list = sub_countries.split(" – ")
        for country in countries_list:
            countries[country.strip()] = {}
    
    return json_friendly_format



In [96]:
def add_area_to_countries(url: str, countries_json_format: dict) -> dict:
    response = requests.get(url)

    if response.status_code != 200:
        raise ValueError('No response from the server!')
    
    soup = BeautifulSoup(response.text, 'html.parser')
    table_container_for_areas = soup.find('table', class_ = "wikitable sortable sticky-header col2left")
    tbody_child_tag = table_container_for_areas.find('tbody')
    area_container = tbody_child_tag.find_all('tr')
    area_by_country_list =  [x.text.strip() for x in area_container[2:]]

    for country_info in area_by_country_list:
        country_cleaned_info = country_info.split('\n')[1:]
        country_name = country_cleaned_info[0].strip()
        total_area_km_2 = re.sub(r"\s*\(.*?\)", "", country_cleaned_info[1])
        land_area_km_2 = re.sub(r"\s*\(.*?\)", "", country_cleaned_info[2])
        water_area_km_2 = re.sub(r"\s*\(.*?\)", "", country_cleaned_info[3])

        if country_name == '9,525,067 (3,677,647)':
            country_name = 'United States'
            total_area_km_2 = '9,525,067'
            land_area_km_2 = '9,147,593'
            water_area_km_2 = '377,424'
        if country_name == '':
            country_name = 'China'
            total_area_km_2 = '9,596,960'
            land_area_km_2 = '9,326,410'
            water_area_km_2 = '270,550'
        if country_name in countries_json_format:       
            countries_json_format[country_name]['total_area_km2'] = total_area_km_2
            countries_json_format[country_name]['land_area_km2'] = land_area_km_2
            countries_json_format[country_name]['water_area_km2'] = water_area_km_2
    
    for country in countries_json_format:
        if 'total_area_km2' not in countries_json_format[country]:
            countries_json_format[country]['total_area_km2'] = None
        if 'land_area_km2' not in countries_json_format[country]:
            countries_json_format[country]['land_area_km2'] = None
        if 'water_area_km2' not in countries_json_format[country]:
            countries_json_format[country]['water_area_km2'] = None

    return countries_json_format

In [97]:
def add_population_to_countries(url: str, countries_json_format: dict) -> dict:
    response = requests.get(url)

    if response.status_code != 200:
        raise ValueError('No response from the server!')
    
    soup = BeautifulSoup(response.text, 'html.parser')
    table_container = soup.find('table', class_='wikitable sortable sticky-header sort-under mw-datatable col2left col6left')
    tbody_child_tag = table_container.find('tbody')
    population_container = tbody_child_tag.find_all('tr')
    population_by_country_list = [x.text.strip() for x in population_container[2:]]

    for country_info in population_by_country_list:
        country_cleaned_info = country_info.split('\n')[2:]
        country_name = country_cleaned_info[0].strip()
        country_population = country_cleaned_info[2].strip()

        if country_name == '1,402,737,000':
            country_name = 'India'
            country_population = '1,402,737,000'
        if country_name in countries_json_format:
            countries_json_format[country_name]['population'] = country_population
    
    for country in countries_json_format:
        if 'population' not in countries_json_format[country]:
             countries_json_format[country]['population'] = None

    return countries_json_format

In [98]:
def add_langueage_to_countries(url: str, countries_json_format: dict) -> dict:
    response = requests.get(url)

    if response.status_code != 200:
        raise ValueError('No response from the server!')
    
    soup = BeautifulSoup(response.text, 'html.parser')
    parent_element = soup.find('div', class_='mw-content-ltr mw-parser-output')
    langueges_tag = parent_element.find_all('p')[3:]
    contry_tag = parent_element.find_all(['ul'])
    # language =  [re.sub(r"\s*\(.*?\)", "", x.text.strip()) for x in langueges_tag]
    # countries =  [a.text for ul in contry_tag for li in ul.find_all('li') for a in li.find_all('a')]

    for p_tag in langueges_tag:
        language = re.sub(r"\s*\(.*?\)", "", p_tag.text.strip())
        next_sibling = p_tag.find_next_sibling('ul')

        if next_sibling:
            countries = [a.text.strip() for li in next_sibling.find_all('li') for a in li.find_all('a')]
            for country in countries:
                if country in countries_json_format:
                    if 'language' in countries_json_format[country]:
                        countries_json_format[country]['language'].append(language)
                    else:
                        countries_json_format[country]['language'] = []
                        countries_json_format[country]['language'].append(language)
    
    for country in countries_json_format:
        if 'language' not in countries_json_format[country]:
            countries_json_format[country]['language'] = []
    return countries_json_format

In [99]:
def add_capital_and_location_to_countries(url: str, countries_json_format: dict) -> dict:
    response = requests.get(url)

    if response.status_code != 200:
        raise ValueError('No response from the server!')
    
    soup = BeautifulSoup(response.text, 'html.parser')
    table_container = soup.find('table', class_='wikitable sortable sticky-header')
    tbody_container = table_container.find('tbody')
    row_container = tbody_container.find_all('tr')
    
    for row in row_container[1:]:
        td_tag = row.find_all('td')
        if len(td_tag) >= 3:
            capital = td_tag[0].find('a').text.strip()
            country = td_tag[1].find('a').text.strip() 
            continent = td_tag[2].text.strip()
            
            if country in countries_json_format:
                if continent:
                    countries_json_format[country]['continent'] = continent
                if capital:
                    countries_json_format[country]['capital'] = capital
    
    for country in countries_json_format:
        if 'continent' not in countries_json_format[country]:
            countries_json_format[country]['continent'] = None
        if 'capital' not in countries_json_format[country]:
            countries_json_format[country]['capital'] = None

    return countries_json_format


In [100]:
def add_minimum_wages_to_countries(url: str, countries_json_format: dict) -> dict:
    response = requests.get(url)

    if response.status_code != 200:
        raise ValueError('No response from the server!')
    
    soup = BeautifulSoup(response.text, 'html.parser')
    table_container = soup.find('table', class_='wikitable sortable mw-datatable static-row-numbers sticky-header-multi sort-under col1left col2left col9left')
    tbody_container = table_container.find('tbody')
    row_container = tbody_container.find_all('tr')

    for row in row_container[1:]:
        td_tag = row.find_all('td')
        if len(td_tag) >= 3:
            country = td_tag[0].find('a').text.strip()
            minimum_weges_per_year = td_tag[2].find('span').text.strip() if td_tag[2].find('span') else None
            
            if country in countries_json_format:
                if minimum_weges_per_year:
                    countries_json_format[country]['minimum_weges_per_year'] = minimum_weges_per_year
    
    for country in countries_json_format:
        if 'minimum_weges_per_year' not in countries_json_format[country]:
            countries_json_format[country]['minimum_weges_per_year'] = None

    return countries_json_format

In [101]:
def add_english_speaking_population_to_countries(url: str, countries_json_format: dict) -> dict:
    response = requests.get(url)

    if response.status_code != 200:
        raise ValueError('No response from the server!')
    
    soup = BeautifulSoup(response.text, 'html.parser')
    table_container = soup.find('table', class_='wikitable sortable')
    tbody_container = table_container.find('tbody')
    row_container = tbody_container.find_all('tr')

    for row in row_container[1:len(row_container)-2]:
        td_tag = row.find_all('td')
        if len(td_tag) >= 4:
            country = td_tag[0].find('a').text.strip() if td_tag[0] else None
            english_speaking_percentage = td_tag[3].text.strip() if td_tag[3] else None

            if country in countries_json_format:
                if english_speaking_percentage:
                    countries_json_format[country]['english_speaking_percentage'] = english_speaking_percentage
    
    for country in countries_json_format:
        if 'english_speaking_percentage' not in countries_json_format[country]:
            countries_json_format[country]['english_speaking_percentage'] = None

    return countries_json_format

In [102]:
def add_fertility_rate_2024_to_countries(url: str, countries_json_format: dict) -> dict:
    response = requests.get(url)

    if response.status_code != 200:
        raise ValueError('No response from the server!')
    
    soup = BeautifulSoup(response.text, 'html.parser')
    table_container = soup.find('table', class_='wikitable sortable sticky-header')
    tbody_container = table_container.find('tbody')
    row_container = tbody_container.find_all('tr')

    for row in row_container[1:]:
        td_tag = row.find_all('td')
        if len(td_tag) >= 3:
            country = td_tag[1].find('a').text.strip() if td_tag[1].find('a') else None
            fertility_rate_2024 = td_tag[2].text.strip() if td_tag[2] else None

            if country in countries_json_format:
                if fertility_rate_2024:
                    countries_json_format[country]['fertility_rate_2024'] = fertility_rate_2024
    
    for country in countries_json_format:
        if 'fertility_rate_2024' not in countries_json_format[country]:
            countries_json_format[country]['fertility_rate_2024'] = None
    
    return countries_json_format
    

In [103]:
def add_iso_2_code_to_countries(url: str, countries_json_format: dict) -> dict:
    response = requests.get(url)

    if response.status_code != 200:
        raise ValueError('No response from the server!')
    
    soup = BeautifulSoup(response.text, 'html.parser')
    table_container = soup.find('table', 'sortable wikitable sticky-header-multi sort-under col1left col2left')
    tbody_container = table_container.find('tbody')
    row_container = tbody_container.find_all('tr')
    for row in row_container[1:]:
        td_tag = row.find_all('td')
        if len(td_tag) >= 4:
            country = td_tag[0].find_all('a')[1].text.strip() if td_tag[0].find('a') else None
            if country:
                country = re.sub(r"\s*\(.*?\)", "", country)
            iso_2 = td_tag[3].find('span').text.strip() if td_tag[3].find('span') else None
            
            if country in countries_json_format:
                if iso_2:
                    countries_json_format[country]['iso_2'] = iso_2
    
    for country in countries_json_format:
        if 'iso_2' not in countries_json_format[country]:
            countries_json_format[country]['iso_2'] = None
    
    return countries_json_format

In [104]:
def add_median_age_to_countries(url: str, contries_json_format: dict) -> dict:
    response = requests.get(url)

    if response.status_code != 200:
        raise ValueError('No response from the server!')
    
    soup = BeautifulSoup(response.text, 'html.parser')
    table_container = soup.find('table', class_='wikitable sortable plainrowheaders')
    tbody_container = table_container.find('tbody')
    row_container = tbody_container.find_all('tr')
    
    for row in row_container[3:]:
        td_tag = row.find_all(['th', 'td'])
        if len(td_tag) >= 3:
            country = td_tag[0].find('a').text.strip() if td_tag[0].find('a') else None
            median_age = td_tag[2].text.strip() if td_tag[2] else None
            
            if country in contries_json_format:
                if median_age:
                    contries_json_format[country]['median_age'] = median_age
    
    for country in contries_json_format:
        if 'median_age' not in contries_json_format[country]:
            contries_json_format[country]['median_age'] = None
    
    return contries_json_format

In [105]:
def add_influation_rate_to_contries(url: str, countries_json_format: dict) -> dict:
    response = requests.get(url)

    if response.status_code != 200:
        raise ValueError('No response from the server!')
    
    
    soup = BeautifulSoup(response.text, 'html.parser')
    table_container = soup.find('table', class_='mw-datatable wikitable sortable sticky-header-multi static-row-numbers sort-under')
    tbody_container = table_container.find('tbody')
    row_container = tbody_container.find_all('tr')
    
    for row in row_container[3:]:
        td_tag = row.find_all('td')
        
        if len(td_tag) >= 15:
            country = td_tag[0].find('a').text.strip() if td_tag[0].find('a') else None
            influation_rate = td_tag[14].text if td_tag[14] else None

        if country in countries_json_format:
            if influation_rate:
                countries_json_format[country]['influation_rate_2024'] = influation_rate
        
    for country in countries_json_format:
        if 'influation_rate_2024' not in countries_json_format[country]:
            countries_json_format[country]['influation_rate_2024'] = None
    
    return countries_json_format    


In [106]:
def add_debt_rate_to_contries(url: str, countries_json_format: dict) -> dict:
    response = requests.get(url)

    if response.status_code != 200:
        raise ValueError('No response from the server!')
    
    
    soup = BeautifulSoup(response.text, 'html.parser')
    table_container = soup.find('table', class_='wikitable sortable sticky-header-multi static-row-numbers sort-under col1left')
    tbody_container = table_container.find('tbody')
    row_container = tbody_container.find_all('tr')
    
    for row in row_container[2:]:
        td_tag = row.find_all('td')
        if len(td_tag) >= 3:
            country = td_tag[0].find('a').text.strip() if td_tag[0].find('a') else None
            debt_rate = td_tag[2].text if td_tag[2] else None

        if country in countries_json_format:
            if debt_rate:
                countries_json_format[country]['total_debt'] = debt_rate
        
    for country in countries_json_format:
        if 'total_debt' not in countries_json_format[country]:
            countries_json_format[country]['total_debt'] = None
    
    return countries_json_format    

In [107]:
def add_electricity_consumption_to_countries(url: str, countries_json_format: dict) -> dict:
    response = requests.get(url)

    if response.status_code != 200:
        raise ValueError('No response from the server!')
    
    
    soup = BeautifulSoup(response.text, 'html.parser')
    table_container = soup.find('table', class_='wikitable sortable sticky-header static-row-numbers sort-under col1left')
    tbody_container = table_container.find('tbody')
    row_container = tbody_container.find_all('tr')
    
    for row in row_container[2:]:
        td_tag = row.find_all('td')

        if len(td_tag) >= 2:
            country = td_tag[0].find('a').text.strip() if td_tag[0].find('a') else None
            consumption_twh = td_tag[1].text.strip() if td_tag[1] else None

        if country in countries_json_format:
            if consumption_twh:
                countries_json_format[country]['electricity_consumption_twh'] = consumption_twh
    
    for country in countries_json_format:
        if 'electricity_consumption_twh' not in countries_json_format[country]:
            countries_json_format[country]['electricity_consumption_twh'] = None
    
    return countries_json_format

In [108]:
def add_average_elevation_to_countries(url: str, countries_json_format: dict) -> dict:
    response = requests.get(url)

    if response.status_code != 200:
        raise ValueError('No response from the server!')
    
    
    soup = BeautifulSoup(response.text, 'html.parser')
    table_container = soup.find('table', class_='wikitable sortable sticky-header-multi static-row-numbers sort-under col1left')
    tbody_container = table_container.find('tbody')
    row_container = tbody_container.find_all('tr')
    
    for row in row_container:
        td_tag = row.find_all('td')
        if len(td_tag) >= 3:
            country = td_tag[0].find('a').text.strip() if td_tag[0].find('a') else None
            average_elevation = td_tag[1].text.strip() if td_tag[1] else None
            average_elevation = re.sub(r"\s*\(.*?\).*", "", average_elevation)

            if country in countries_json_format:
                if average_elevation:
                    countries_json_format[country]['average_elevation'] = average_elevation
    
    for country in countries_json_format:
        if 'average_elevation' not in countries_json_format[country]:
            countries_json_format[country]['average_elevation'] = None
    
    return countries_json_format

In [109]:
countries = create_json_friendly_format_structure_with_all_countries(url_list_of_countries, countries)
countries = add_area_to_countries(url_list_of_areas, countries)
countries = add_population_to_countries(url_list_of_population, countries)
countries = add_langueage_to_countries(url_list_of_languages, countries)
countries = add_capital_and_location_to_countries(url_list_of_location_and_capital, countries)
countries = add_minimum_wages_to_countries(url_list_of_minimum_wages, countries)
countries = add_english_speaking_population_to_countries(url_list_of_english_speaking_by_percentage, countries)
countries = add_fertility_rate_2024_to_countries(url_list_fertility_rate_2024, countries)
countries = add_iso_2_code_to_countries(url_list_country_iso_2, countries)
countries = add_median_age_to_countries(url_list_median_age, countries)
countries = add_influation_rate_to_contries(url_list_inflation_rate, countries)
countries = add_debt_rate_to_contries(url_list_debt, countries)
countries = add_electricity_consumption_to_countries(url_list_electricity_consumption, countries)
countries = add_average_elevation_to_countries(url_list_average_elevation, countries)

file_path = 'countries_data.json'
with open(file_path, 'w') as j:
    json.dump(countries, j, indent=4)

In [111]:
df = pd.DataFrame(countries)
df = df.T
df = df.reset_index(names='countries')
df.head()

,countries,total_area_km2,land_area_km2,water_area_km2,population,language,continent,capital,minimum_weges_per_year,english_speaking_percentage,fertility_rate_2024,iso_2,median_age,influation_rate_2024,total_debt,electricity_consumption_twh,average_elevation
0,Afghanistan,"652,864","652,230",630,"42,045,000","[Dari:, Pashto:, Pashto:, Persian:, Persian:]",Asia,Kabul,858,6,4.3,AF,19.9,None,1.28 billion,7.19,"1,884"
1,Albania,"28,748","27,400",330,"2,402,113","[Albanian:, Greek:, Greek:, Macedonian]",Europe,Tirana,"4,637",None,1.4,AL,35.8,None,10.8 billion,8.09,708
2,Algeria,"2,381,741","2,381,741",0,"47,400,000","[Arabic:, Berber:]",Africa,Algiers,"1,777",7,2.7,DZ,28.9,9.32,3.7 billion,94.02,800
3,Andorra,468,468,0,"87,097",[Catalan:],None,Andorra la Vella,"18,253",22,-,AD,48.1,None,1.11 billion,None,"1,996"
4,Angola,"1,246,700","1,246,700",0,"35,121,734",[Portuguese:],Africa,Luanda,663,None,5.0,AO,16.2,13.64,37.7 billion,17.94,"1,112"
